# P4 Depth Completion — Full-Resource Kaggle Experiments

**Branch:** `feat/research-driven-depth-completion`
**Run top-to-bottom on Kaggle T4 x2 GPU runtime.**

## Experiment map
| ID | Description |
|---|---|
| EXP-001 | Interpolation baselines x4 clean n=654 (full val) |
| EXP-002 | Classical blend vs bicubic x4 clean n=654 |
| EXP-003 | SGNet deep x4 clean n=654 |
| EXP-004 | Hybrid v2 x4 clean n=654 |
| EXP-005 | Fixed-alpha ablation sweep x4 clean n=200 |
| EXP-006 | Consistency-strength sweep x4 clean n=100 |
| EXP-007 | Robustness: holes 5/10/20/30% x4 n=100 each |
| EXP-008 | Robustness: noise std 0.01/0.02/0.05/0.10 x4 n=100 each |
| EXP-009 | Full scale matrix x2/x4/x8/x16 all methods n=200 |
| EXP-010 | hard_overwrite ablation (PENet-style) x4 n=100 |
| EXP-011 | Regime detection accuracy: holes+noise x4 n=100 |
| EXP-012 | Combined degradation: holes+noise x4 n=100 |
| EXP-D   | Official hackathon testset x2/x4/x8 all methods |

**Honesty protocol:** deep/hybrid reported *not evaluated* when checkpoint missing, never silently scored.
Metrics: RMSE / MAE / absRel / delta1 / iRMSE / iMAE / RMSE-hole / PSNR / SSIM / boundary-RMSE / edge-F1 / latency


In [ ]:
#@title Cell 1 — Environment
import sys, platform, torch
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print(f"GPUs available: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/2**30:.1f} GiB")
import psutil
print(f"RAM: {psutil.virtual_memory().total/2**30:.1f} GiB total, "
      f"{psutil.virtual_memory().available/2**30:.1f} GiB free")
import os
print(f"CPUs: {os.cpu_count()}")


In [ ]:
#@title Cell 2 — Clone repo and install dependencies
import os, subprocess, sys, shutil

REPO   = "/kaggle/working/CV-Hackthon"
BRANCH = "feat/research-driven-depth-completion"

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/KumarSrinidhi/CV-Hackthon.git", REPO],
                   check=True)

os.chdir(REPO)

# remove any untracked results/ files left by a previous run on master
# so they don't block the branch checkout
results_dir = os.path.join(REPO, "results")
if os.path.isdir(results_dir):
    shutil.rmtree(results_dir)
    os.makedirs(results_dir)
    print("Cleared stale results/ directory.")

subprocess.run(["git", "fetch", "-q", "--all"], check=True)
subprocess.run(["git", "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "pull", "-q", "origin", BRANCH], check=True)

reqs = [l.strip() for l in open("requirements.txt")
        if l.strip() and not l.startswith("#")]
reqs = [r for r in reqs if not r.lower().startswith(("torch", "torchvision"))]
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *reqs, "h5py", "huggingface_hub", "psutil"], check=True)
sys.path.insert(0, REPO)

branch = subprocess.check_output(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"repo: {os.getcwd()}")
print(f"branch: {branch}  commit: {commit}")
assert branch == BRANCH, f"ERROR: expected {BRANCH}, got {branch}"


## Checkpoints

Attach a Kaggle dataset containing `SGNet_X4_R.pth`, `SGNet_X8_R.pth`, `SGNet_X16_R.pth`
**flat in the dataset root** (no subfolders), then set `CKPT_DIR` below.
Without checkpoints, `deep`/`hybrid*` are reported as *not evaluated*.


In [ ]:
#@title Cell 3 — Checkpoint status
CKPT_DIR = ""  #@param {type:"string"}

import os, glob
if CKPT_DIR.strip():
    os.environ["SGNET_CKPT_DIR"] = CKPT_DIR.strip()
else:
    # auto-detect from attached datasets
    hits = glob.glob("/kaggle/input/**/*.pth", recursive=True)
    if hits:
        detected = str(__import__("pathlib").Path(hits[0]).parent)
        os.environ["SGNET_CKPT_DIR"] = detected
        print(f"Auto-detected checkpoint dir: {detected}")

from deep.settings import ckpt_dir
from deep.deep_model import resolve_checkpoint

print("checkpoint dir:", ckpt_dir())
CKPT_AVAILABLE = {}
for s in (4, 8, 16):
    try:
        p = resolve_checkpoint(s)
        CKPT_AVAILABLE[s] = str(p)
        print(f"  x{s}: {p.name}  ({p.stat().st_size/2**20:.0f} MB)")
    except Exception as e:
        CKPT_AVAILABLE[s] = None
        print(f"  x{s}: NOT AVAILABLE ({type(e).__name__}: {e})")


## Data — HF NYU val tar

With Internet ON the next cell downloads from HuggingFace automatically (~1 GB).
Alternatively attach a Kaggle dataset containing `val-000000.tar` and set `TAR_PATH`.


In [ ]:
#@title Cell 4 — Locate data: tar file OR loose .h5 folder + cache to RAM disk
import glob, os, shutil, tarfile, io
from pathlib import Path

# ── Option A: point directly at a tar ──────────────────────────────────────
TAR_PATH = ""  #@param {type:"string"}

# ── Option B: point at the folder containing .h5 files ─────────────────────
H5_DIR = ""  #@param {type:"string"}

# ── auto-detect ─────────────────────────────────────────────────────────────
def _find_tar():
    hits = glob.glob("/kaggle/input/**/val-*.tar", recursive=True)
    return hits[0] if hits else ""

def _find_h5_dir():
    # search common Kaggle dataset mount patterns for .h5 files
    hits = glob.glob("/kaggle/input/**/*.h5", recursive=True)
    if hits:
        return str(Path(hits[0]).parent)
    return ""

TAR = None
H5_FOLDER = None

if TAR_PATH.strip():
    TAR = TAR_PATH.strip()
elif H5_DIR.strip():
    H5_FOLDER = H5_DIR.strip()
elif _find_tar():
    TAR = _find_tar()
    print(f"Auto-detected tar: {TAR}")
elif _find_h5_dir():
    H5_FOLDER = _find_h5_dir()
    print(f"Auto-detected .h5 folder: {H5_FOLDER}")
else:
    # fallback: download from HuggingFace
    from huggingface_hub import HfApi, hf_hub_download
    files = HfApi().list_repo_files("sayakpaul/nyu_depth_v2", repo_type="dataset")
    cands = [f for f in files if f.endswith(".tar")]
    val_cands = [f for f in cands if "val" in os.path.basename(f).lower()] or cands
    TAR = hf_hub_download("sayakpaul/nyu_depth_v2", repo_type="dataset",
                          filename=sorted(val_cands)[0])
    print(f"Downloaded tar: {TAR}")

# ── if we have a folder of .h5 files, pack them into a RAM-disk tar ─────────
if H5_FOLDER:
    h5_files = sorted(glob.glob(os.path.join(H5_FOLDER, "**/*.h5"), recursive=True) +
                      glob.glob(os.path.join(H5_FOLDER, "*.h5")))
    print(f"Found {len(h5_files)} .h5 files in {H5_FOLDER}")
    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found under {H5_FOLDER}")
    RAM_TAR = "/dev/shm/val-assembled.tar"
    print(f"Packing {len(h5_files)} .h5 files into RAM tar {RAM_TAR} ...")
    with tarfile.open(RAM_TAR, "w") as tf:
        for p in h5_files:
            tf.add(p, arcname=Path(p).name)
    print(f"RAM tar ready: {os.path.getsize(RAM_TAR)/1e6:.0f} MB")
    TAR = RAM_TAR

# ── if we have a tar on disk, cache it to RAM ────────────────────────────────
if TAR and not TAR.startswith("/dev/shm"):
    RAM_TAR = "/dev/shm/val-000000.tar"
    if not os.path.exists(RAM_TAR):
        print(f"Copying tar to RAM disk ({os.path.getsize(TAR)/1e6:.0f} MB) ...")
        shutil.copy2(TAR, RAM_TAR)
    TAR = RAM_TAR

print(f"\nUsing TAR: {TAR}  ({os.path.getsize(TAR)/1e6:.0f} MB)")

# quick sanity check
import tarfile as _tf
with _tf.open(TAR) as t:
    h5s = [m.name for m in t if m.name.endswith(".h5")]
print(f"Samples in tar: {len(h5s)}  (first: {h5s[0] if h5s else 'none'})")


In [ ]:
#@title Cell 5 — Preload SGNet on both GPUs (warm cache, zero cold-start per experiment)
import torch, os
from deep.deep_model import get_model, clear_model_cache

N_GPU = torch.cuda.device_count()
print(f"Preloading models on {N_GPU} GPU(s)...")

LOADED = {}  # (scale, gpu_id) -> info
for gpu_id in range(N_GPU):
    dev = torch.device(f"cuda:{gpu_id}")
    for scale in (4, 8, 16):
        if CKPT_AVAILABLE.get(scale):
            try:
                _, info = get_model(scale, dev)
                LOADED[(scale, gpu_id)] = info
                mem = torch.cuda.memory_allocated(dev) / 2**20
                print(f"  cuda:{gpu_id} x{scale}: {info['params_m']}M params  "
                      f"({info['size_mb_fp32']} MB fp32)  alloc={mem:.0f} MB")
            except Exception as e:
                print(f"  cuda:{gpu_id} x{scale}: FAILED {e}")
        else:
            print(f"  cuda:{gpu_id} x{scale}: skipped (no checkpoint)")

# Set default device for inference to cuda:0
os.environ.setdefault("SGNET_DEVICE", "cuda:0")
print(f"\nModels loaded: {list(LOADED.keys())}")
print("GPU 0 memory allocated:", torch.cuda.memory_allocated(0)/2**20, "MB")
if N_GPU > 1:
    print("GPU 1 memory allocated:", torch.cuda.memory_allocated(1)/2**20, "MB")


In [ ]:
#@title Cell 6 — Parallel experiment runner (uses both GPUs simultaneously)
import os, threading, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

N_GPU = torch.cuda.device_count()

def run_exp(tag, methods, scale=4, deg=None, max_samples=100, gpu_id=0, out_dir=None):
    """Run one experiment on a specific GPU, save results, return (tag, markdown)."""
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    deg = deg or DegradationConfig()
    ds  = HFNYUValDataset(scale, deg, tar_paths=[TAR])
    res = evaluate_methods(ds, methods, max_samples=max_samples)
    out = out_dir or f"results/kaggle_{tag}"
    save_results(res, out, tag)
    return tag, summary_markdown(res)

def run_parallel(jobs):
    """jobs = list of (tag, methods, scale, deg, max_samples, gpu_id, out_dir)
    Runs pairs concurrently across GPUs, prints results as they finish."""
    results = {}
    # split into pairs for 2-GPU parallelism
    for i in range(0, len(jobs), N_GPU):
        batch = jobs[i:i+N_GPU]
        threads = []
        out_box = {}
        def _run(j, box=out_box):
            tag, md_str = run_exp(*j)
            box[tag] = md_str
        for j in batch:
            t = threading.Thread(target=_run, args=(j,))
            t.start()
            threads.append(t)
        for t in threads:
            t.join()
        for tag, md_str in out_box.items():
            print(f"\n{'='*60}\n{tag}\n{'='*60}")
            print(md_str)
            results[tag] = md_str
    return results

print("Runner ready. N_GPU =", N_GPU)


## EXP-001 — Interpolation baselines (full val, n=654)

Nearest / bilinear / bicubic at x4 clean on the **full** HF NYU val set.
These are the definitive reference numbers.


In [ ]:
#@title EXP-001 — baselines x4 clean n=654
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
print(f"Dataset size: {len(ds)} samples")
res = evaluate_methods(ds, ["nearest", "bilinear", "bicubic"], max_samples=654)
save_results(res, "results/kaggle_exp001_baselines", "exp001_baselines_x4")
print(summary_markdown(res))


## EXP-002 — Improved classical vs bicubic (n=654)

`classical_hr_depth()` defaults to `classical_blend`: iterative bicubic back-projection
blended 40% with edge-sharpened dense path. Prior local validation: RMSE 0.0250 vs 0.0292.


In [ ]:
#@title EXP-002 — classical_blend vs bicubic x4 clean n=654
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["bicubic", "classical"], max_samples=654)
save_results(res, "results/kaggle_exp002_classical", "exp002_classical_x4")
print(summary_markdown(res))


## EXP-003 — SGNet deep branch x4 (n=654)

Full val set. Regime-aware adaptive checkpoint selection active.
Requires `SGNet_X4_R.pth`.


In [ ]:
#@title EXP-003 — SGNet deep x4 clean n=654
import os
os.environ["SGNET_DEVICE"] = "cuda:0"
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["bicubic", "deep"], max_samples=654)
save_results(res, "results/kaggle_exp003_deep", "exp003_deep_x4")
print(summary_markdown(res))


## EXP-004 — Hybrid v2 x4 (n=654)

Regime-aware fusion + hole-band confidence boost. Full val set.


In [ ]:
#@title EXP-004 — hybrid v2 x4 clean n=654
import os
os.environ["SGNET_DEVICE"] = "cuda:0"
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["bicubic", "deep", "hybrid"], max_samples=654)
save_results(res, "results/kaggle_exp004_hybrid", "exp004_hybrid_x4")
print(summary_markdown(res))


## EXP-005 — Fixed-alpha ablation sweep x4 (n=200, both GPUs)

Sweep alpha in {0.0, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.50} with and without
measurement consistency. Runs alpha groups in parallel across both GPUs.


In [ ]:
#@title EXP-005 — fixed-alpha sweep x4 n=200 (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ALPHAS = [0.0, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.50]
N_GPU  = torch.cuda.device_count()

# build method lists: no-consistency on gpu0, with-consistency on gpu1
no_cons  = ["deep"] + [f"fixed_{a}" for a in ALPHAS]
with_cons = [f"fixed_consistency_{a}" for a in ALPHAS]

results_box = {}

def _run(methods, gpu_id, tag):
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
    res = evaluate_methods(ds, methods, max_samples=200)
    save_results(res, f"results/kaggle_exp005_{tag}", f"exp005_{tag}")
    results_box[tag] = summary_markdown(res)

t0 = threading.Thread(target=_run, args=(no_cons,  0, "no_consistency"))
t1 = threading.Thread(target=_run, args=(with_cons, 1, "with_consistency"))
t0.start(); t1.start()
t0.join();  t1.join()

for tag in ("no_consistency", "with_consistency"):
    print(f"\n--- EXP-005 {tag} ---")
    print(results_box[tag])


## EXP-006 — Consistency-strength sweep x4 (n=100, both GPUs)

Sweep consistency strength in {0.05, 0.10, 0.15, 0.25, 0.35, 0.50} at 1 and 2 iterations.
GPU0: 1-iteration variants. GPU1: 2-iteration variants.


In [ ]:
#@title EXP-006 — consistency-strength sweep x4 n=100 (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

STRENGTHS = [0.05, 0.10, 0.15, 0.25, 0.35, 0.50]
N_GPU = torch.cuda.device_count()

iter1_methods = ["deep"] + [f"dl_consistency_s{s}_i1" for s in STRENGTHS]
iter2_methods = ["deep"] + [f"dl_consistency_s{s}_i2" for s in STRENGTHS]

results_box = {}

def _run(methods, gpu_id, tag):
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
    res = evaluate_methods(ds, methods, max_samples=100)
    save_results(res, f"results/kaggle_exp006_{tag}", f"exp006_{tag}")
    results_box[tag] = summary_markdown(res)

t0 = threading.Thread(target=_run, args=(iter1_methods, 0, "iter1"))
t1 = threading.Thread(target=_run, args=(iter2_methods, 1, "iter2"))
t0.start(); t1.start()
t0.join();  t1.join()

for tag in ("iter1", "iter2"):
    print(f"\n--- EXP-006 {tag} ---")
    print(results_box[tag])


## EXP-007 — Robustness: holes sweep x4 (n=100 each, both GPUs)

hole_ratio in {0.05, 0.10, 0.20, 0.30}. GPU0: 5%+20%, GPU1: 10%+30%.
Tests whether hybrid v2 hole-band confidence recovers deep's degradation under missing pixels.


In [ ]:
#@title EXP-007 — holes robustness sweep x4 n=100 each (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["bicubic", "classical", "deep", "hybrid", "hybrid_fixed02"]
N_GPU   = torch.cuda.device_count()
HOLE_RATIOS = [0.05, 0.10, 0.20, 0.30]
results_box = {}

def _run(ratio, gpu_id):
    tag = f"holes_{int(ratio*100):02d}pct"
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(4, DegradationConfig(hole_ratio=ratio), tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=100)
    save_results(res, f"results/kaggle_exp007_{tag}", f"exp007_{tag}")
    results_box[tag] = summary_markdown(res)

# GPU0: 5% and 20%, GPU1: 10% and 30% (sequential within each GPU)
def _gpu0():
    _run(0.05, 0)
    _run(0.20, 0)

def _gpu1():
    _run(0.10, 1)
    _run(0.30, 1)

t0 = threading.Thread(target=_gpu0)
t1 = threading.Thread(target=_gpu1)
t0.start(); t1.start()
t0.join();  t1.join()

for ratio in HOLE_RATIOS:
    tag = f"holes_{int(ratio*100):02d}pct"
    print(f"\n--- EXP-007 {tag} ---")
    print(results_box[tag])


## EXP-008 — Robustness: noise sweep x4 (n=100 each, both GPUs)

noise_std in {0.01, 0.02, 0.05, 0.10} m. GPU0: 0.01+0.05, GPU1: 0.02+0.10.


In [ ]:
#@title EXP-008 — noise robustness sweep x4 n=100 each (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["bicubic", "classical", "deep", "hybrid", "hybrid_fixed02"]
N_GPU   = torch.cuda.device_count()
NOISE_STDS = [0.01, 0.02, 0.05, 0.10]
results_box = {}

def _run(std, gpu_id):
    tag = f"noise_std{str(std).replace('.','p')}"
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(4, DegradationConfig(noise_std=std), tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=100)
    save_results(res, f"results/kaggle_exp008_{tag}", f"exp008_{tag}")
    results_box[tag] = summary_markdown(res)

def _gpu0():
    _run(0.01, 0)
    _run(0.05, 0)

def _gpu1():
    _run(0.02, 1)
    _run(0.10, 1)

t0 = threading.Thread(target=_gpu0)
t1 = threading.Thread(target=_gpu1)
t0.start(); t1.start()
t0.join();  t1.join()

for std in NOISE_STDS:
    tag = f"noise_std{str(std).replace('.','p')}"
    print(f"\n--- EXP-008 {tag} ---")
    print(results_box[tag])


## EXP-009 — Full scale matrix x2/x4/x8/x16 (n=200, both GPUs)

All methods across all supported scales. x4+x16 on GPU0, x2+x8 on GPU1 (concurrent).
x2 is classical-only (no SGNet checkpoint). x16 uses SGNet_X16_R.pth.


In [ ]:
#@title EXP-009 — full scale matrix x2/x4/x8/x16 n=200 (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["nearest", "bilinear", "bicubic", "classical",
           "deep", "hybrid", "hybrid_fixed02", "hybrid_adaptive"]
N_GPU   = torch.cuda.device_count()
results_box = {}

def _run_scale(scale, gpu_id):
    tag = f"scale_x{scale}"
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(scale, DegradationConfig(), tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=200)
    save_results(res, f"results/kaggle_exp009_{tag}", f"exp009_{tag}")
    results_box[tag] = summary_markdown(res)

def _gpu0():   # x4 then x16
    _run_scale(4,  0)
    _run_scale(16, 0)

def _gpu1():   # x2 then x8
    _run_scale(2, 1)
    _run_scale(8, 1)

t0 = threading.Thread(target=_gpu0)
t1 = threading.Thread(target=_gpu1)
t0.start(); t1.start()
t0.join();  t1.join()

for scale in (2, 4, 8, 16):
    tag = f"scale_x{scale}"
    print(f"\n=== EXP-009 x{scale} ===")
    print(results_box.get(tag, "not run"))


## EXP-010 — hard_overwrite ablation (PENet-style per-pixel re-injection)

`enforce_consistency(hard_overwrite=True)` hard-overwrites HR pixels at valid LR
locations with nearest-upsampled LR values. Tests whether this helps or hurts vs
the soft damped correction. Both GPUs used: GPU0=clean, GPU1=holes.


In [ ]:
#@title EXP-010 — hard_overwrite ablation x4 n=100 (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown
from hybrid.pipeline import hybrid_hr_depth
from hybrid.consistency import enforce_consistency

N_GPU = torch.cuda.device_count()
results_box = {}

def _hybrid_hard(lr, rgb, mask, scale=4):
    pred = hybrid_hr_depth(lr, rgb, mask, scale, use_consistency=False)
    import numpy as np
    from data.preprocessing import fill_invalid_nearest, make_valid_mask
    valid = make_valid_mask(lr, mask)
    import cv2
    lr_filled = fill_invalid_nearest(lr, valid)
    lr_up = cv2.resize(lr_filled, (pred.shape[1], pred.shape[0]),
                       interpolation=cv2.INTER_NEAREST)
    return enforce_consistency(pred, lr, valid, scale,
                               strength=0.25, hard_overwrite=True)

def _run(methods, custom, gpu_id, tag, deg):
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(4, deg, tar_paths=[TAR])
    res = evaluate_methods(ds, methods, max_samples=100, methods=custom)
    save_results(res, f"results/kaggle_exp010_{tag}", f"exp010_{tag}")
    results_box[tag] = summary_markdown(res)

base_methods = ["bicubic", "deep", "hybrid", "hybrid_fixed02"]
custom_methods = {"hybrid_hard_overwrite": _hybrid_hard}
all_methods = base_methods + ["hybrid_hard_overwrite"]

t0 = threading.Thread(target=_run, args=(
    all_methods, custom_methods, 0, "clean", DegradationConfig()))
t1 = threading.Thread(target=_run, args=(
    all_methods, custom_methods, 1, "holes10pct",
    DegradationConfig(hole_ratio=0.10)))
t0.start(); t1.start()
t0.join();  t1.join()

for tag in ("clean", "holes10pct"):
    print(f"\n--- EXP-010 {tag} ---")
    print(results_box[tag])


## EXP-011 — Combined degradation: holes + noise simultaneously (n=100, both GPUs)

Real sensors have both missing pixels AND noise. Tests:
- holes=10% + noise=0.02 (GPU0)
- holes=20% + noise=0.05 (GPU1)


In [ ]:
#@title EXP-011 — combined holes+noise x4 n=100 (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["bicubic", "classical", "deep", "hybrid", "hybrid_fixed02"]
N_GPU   = torch.cuda.device_count()
results_box = {}

def _run(hole_ratio, noise_std, gpu_id):
    tag = f"holes{int(hole_ratio*100)}pct_noise{str(noise_std).replace('.','p')}"
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(4, DegradationConfig(hole_ratio=hole_ratio, noise_std=noise_std),
                          tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=100)
    save_results(res, f"results/kaggle_exp011_{tag}", f"exp011_{tag}")
    results_box[tag] = summary_markdown(res)

t0 = threading.Thread(target=_run, args=(0.10, 0.02, 0))
t1 = threading.Thread(target=_run, args=(0.20, 0.05, 1))
t0.start(); t1.start()
t0.join();  t1.join()

for tag in results_box:
    print(f"\n--- EXP-011 {tag} ---")
    print(results_box[tag])


## EXP-012 — Per-scale robustness: x8 and x16 under holes (n=100, both GPUs)

x4 robustness is covered by EXP-007/008. This tests whether the hole-band
confidence boost also helps at x8 and x16.
GPU0: x8 holes 10%, GPU1: x16 holes 10%.


In [ ]:
#@title EXP-012 — per-scale robustness x8/x16 holes 10% n=100 (parallel)
import threading, os, torch
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["bicubic", "classical", "deep", "hybrid"]
N_GPU   = torch.cuda.device_count()
results_box = {}

def _run(scale, gpu_id):
    tag = f"x{scale}_holes10pct"
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = HFNYUValDataset(scale, DegradationConfig(hole_ratio=0.10), tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=100)
    save_results(res, f"results/kaggle_exp012_{tag}", f"exp012_{tag}")
    results_box[tag] = summary_markdown(res)

t0 = threading.Thread(target=_run, args=(8,  0))
t1 = threading.Thread(target=_run, args=(16, 1))
t0.start(); t1.start()
t0.join();  t1.join()

for tag in results_box:
    print(f"\n--- EXP-012 {tag} ---")
    print(results_box[tag])


## EXP-D — Official hackathon testset (20 samples, native missing depth)

`data/depthsr_testset/` from branch `kumarbro`. Methods receive native LR depth + mask.
Hybrid runs with `use_consistency=False` (operator is block-mean, not bicubic).
x2/x4/x8 evaluated; x4+x8 run in parallel across both GPUs.


In [ ]:
#@title EXP-D step 1 — fetch testset from kumarbro branch
import subprocess
subprocess.run(["git", "fetch", "-q", "origin", "kumarbro"], check=True)
subprocess.run(["git", "checkout", "-q", "FETCH_HEAD", "--", "data/depthsr_testset"],
               check=True)
import os
samples = sorted(d for d in os.listdir("data/depthsr_testset")
                 if os.path.isdir(os.path.join("data/depthsr_testset", d)))
print(f"{len(samples)} samples:", samples)


In [ ]:
#@title EXP-D step 2 — evaluate testset x2/x4/x8 (x4+x8 parallel)
import os, threading, numpy as np
from PIL import Image
from data.dataset import DepthSRDataset
from data.preprocessing import to_float_rgb
from eval.evaluate import evaluate_methods, save_results, summary_markdown
from hybrid.pipeline import hybrid_hr_depth
from deep.deep_model import resolve_checkpoint
import torch

N_GPU = torch.cuda.device_count()

class DepthSRTestset(DepthSRDataset):
    name = "depthsr_testset"
    unit = "m"
    data_range = 10.0

    def __init__(self, scale, root="data/depthsr_testset"):
        super().__init__(scale)
        self.folders = sorted(
            os.path.join(root, d) for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )

    def _num_raw(self):
        return len(self.folders)

    def _load_raw(self, i):
        d = self.folders[i]
        rgb = np.asarray(Image.open(os.path.join(d, "rgb.png")))
        gt  = np.load(os.path.join(d, "depth_hr.npy")).astype(np.float32)
        return os.path.basename(d), to_float_rgb(rgb), gt, np.isfinite(gt) & (gt > 0)

    def __getitem__(self, i):
        name, rgb, gt, valid = self._load_raw(i)
        d    = self.folders[i]
        lr   = np.load(os.path.join(d, f"lr_depth_{self.scale}x.npy")).astype(np.float32)
        mask = np.load(os.path.join(d, f"lr_mask_{self.scale}x.npy")).astype(bool)
        return {"name": name,
                "hr_rgb": np.ascontiguousarray(rgb, np.float32),
                "depth_original": gt, "valid_mask_original": valid,
                "lr_depth": lr, "lr_valid_mask": mask, "lr_depth_clean": lr.copy(),
                "scale": self.scale, "unit": self.unit,
                "data_range": float(self.data_range), "operator_exact": False}

def _hybrid_testset(lr, rgb, m, s=4):
    try:
        resolve_checkpoint(s)
    except Exception as e:
        raise NotImplementedError(f"deep checkpoint unavailable: {e}") from e
    return hybrid_hr_depth(lr, rgb, m, s, use_consistency=False)

METHODS = ["nearest", "bilinear", "bicubic", "classical",
           "deep", "hybrid", "hybrid_fixed02", "hybrid_adaptive"]
custom  = {"hybrid": _hybrid_testset}
results_box = {}

def _run_testset(scale, gpu_id):
    os.environ["SGNET_DEVICE"] = f"cuda:{gpu_id % N_GPU}"
    ds  = DepthSRTestset(scale)
    res = evaluate_methods(ds, METHODS, methods=custom)
    save_results(res, f"results/kaggle_testset_x{scale}", f"testset_x{scale}")
    results_box[f"x{scale}"] = summary_markdown(res)

# x2 first (CPU, no GPU contention), then x4+x8 in parallel
_run_testset(2, 0)

t0 = threading.Thread(target=_run_testset, args=(4, 0))
t1 = threading.Thread(target=_run_testset, args=(8, 1))
t0.start(); t1.start()
t0.join();  t1.join()

for scale in (2, 4, 8):
    print(f"\n=== EXP-D testset x{scale} ===")
    print(results_box.get(f"x{scale}", "not run"))


## Results summary

Every experiment writes to `results/kaggle_*/`:
- `*_summary.md` — printed table
- `*_summary.json` — machine-readable
- `*_per_image.csv` — per-sample metrics + latency

Use the FINAL DUMP cell below to get everything in one JSON block.


In [ ]:
#@title Print all summary tables
from IPython.display import Markdown, display
import glob

for md_path in sorted(glob.glob("results/kaggle_*/*summary.md") +
                      glob.glob("results/testset*/*summary.md")):
    print(f"\n{'='*60}\n{md_path}\n{'='*60}")
    display(Markdown(open(md_path).read()))


---
# Classical Branch Experiments (EXP-C01 to EXP-C30+)

All classical experiments are CPU-only (no checkpoint needed).
They run on both CPUs in parallel using `ThreadPoolExecutor`.
Each experiment saves to `results/kaggle_expC*/`.

| ID | Description |
|---|---|
| EXP-C01 | All 9 classical methods head-to-head x4 clean n=200 |
| EXP-C02 | JBU color_thresh sweep x4 n=100 |
| EXP-C03 | JBU radius sweep x4 n=100 |
| EXP-C04 | JBU depth_thresh sweep (adaptive_jbu) x4 n=100 |
| EXP-C05 | Guided filter radius sweep x4 n=100 |
| EXP-C06 | Guided filter eps sweep x4 n=100 |
| EXP-C07 | classical_blend gain sweep x4 n=100 |
| EXP-C08 | Back-projection iters sweep x4 n=100 |
| EXP-C09 | Back-projection gain sweep x4 n=100 |
| EXP-C10 | Boundary refinement relative_jump sweep x4 n=100 |
| EXP-C11 | Boundary refinement sigma_color sweep x4 n=100 |
| EXP-C12 | All methods x2 clean n=200 |
| EXP-C13 | All methods x8 clean n=200 |
| EXP-C14 | All methods x16 clean n=200 |
| EXP-C15 | All methods x4 holes 10% n=100 |
| EXP-C16 | All methods x4 holes 20% n=100 |
| EXP-C17 | All methods x4 noise 0.02 n=100 |
| EXP-C18 | All methods x4 noise 0.05 n=100 |
| EXP-C19 | All methods x4 combined holes10%+noise0.02 n=100 |
| EXP-C20 | JBU radius x scale interaction (r=1,2,3 x scale=2,4,8) n=50 |
| EXP-C21 | guided+boundary_refinement pipeline variants x4 n=100 |
| EXP-C22 | classical_blend gain sweep under holes 10% x4 n=100 |
| EXP-C23 | classical_blend gain sweep under noise 0.02 x4 n=100 |
| EXP-C24 | JBU color_thresh x depth_thresh grid x4 n=50 |
| EXP-C25 | fill_small_holes relative_span sweep x4 n=100 |
| EXP-C26 | All methods full val n=654 x4 clean (definitive classical numbers) |
| EXP-C27 | Per-scale best classical vs bicubic (x2/x4/x8/x16) n=200 |
| EXP-C28 | Holes sweep 5/10/20/30/50% best classical x4 n=100 |
| EXP-C29 | Noise sweep 0.01/0.02/0.05/0.10/0.20 best classical x4 n=100 |
| EXP-C30 | Combined degradation grid (holes x noise) best classical x4 n=50 |


In [ ]:
#@title Classical runner helpers (CPU parallel, no GPU needed)
import os, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown
from classical.classical_hr_depth import reconstruct

def _classical_method(name, **kwargs):
    """Return a callable for reconstruct(method=name, **kwargs)."""
    def fn(lr, rgb, mask, scale=4):
        return reconstruct(lr, rgb, mask, scale, method=name, **kwargs)
    fn.__name__ = name
    return fn

def run_classical(tag, methods, custom=None, scale=4, deg=None,
                  max_samples=100, out_dir=None):
    deg = deg or DegradationConfig()
    ds  = HFNYUValDataset(scale, deg, tar_paths=[TAR])
    res = evaluate_methods(ds, methods, max_samples=max_samples,
                           methods=custom or {})
    out = out_dir or f"results/kaggle_{tag}"
    save_results(res, out, tag)
    return tag, summary_markdown(res)

def run_classical_parallel(jobs, max_workers=4):
    """jobs = list of (tag, methods, custom, scale, deg, max_samples, out_dir)"""
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_classical, *j): j[0] for j in jobs}
        for f in as_completed(futs):
            tag, md_str = f.result()
            results[tag] = md_str
            print(f"  done: {tag}")
    return results

print("Classical runner ready.")


## EXP-C01 — All 9 classical methods head-to-head x4 clean n=200


In [ ]:
#@title EXP-C01 — all classical methods x4 clean n=200
tag, md_str = run_classical(
    "expC01_all_methods_x4",
    ["nearest","bilinear","bicubic","jbu","guided",
     "classical","classical_blend","adaptive_jbu","refined_jbu"],
    scale=4, max_samples=200
)
print(md_str)


## EXP-C02 — JBU color_thresh sweep x4 n=100

Sweep `color_thresh` in {0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20}.


In [ ]:
#@title EXP-C02 — JBU color_thresh sweep x4 n=100 (parallel)
THRESHOLDS = [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]
jobs = []
custom_all = {}
methods_list = []
for t in THRESHOLDS:
    name = f"jbu_ct{str(t).replace('.','p')}"
    custom_all[name] = _classical_method("jbu", color_thresh=t)
    methods_list.append(name)

jobs = [("expC02_jbu_color_thresh", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C03 — JBU radius sweep x4 n=100

Sweep `radius` in {1, 2, 3, 4, 5}.


In [ ]:
#@title EXP-C03 — JBU radius sweep x4 n=100 (parallel)
RADII = [1, 2, 3, 4, 5]
custom_all = {}
methods_list = []
for r in RADII:
    name = f"jbu_r{r}"
    custom_all[name] = _classical_method("jbu", radius=r)
    methods_list.append(name)

jobs = [("expC03_jbu_radius", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C04 — Adaptive JBU depth_thresh sweep x4 n=100

Sweep `depth_thresh` in {0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30}.


In [ ]:
#@title EXP-C04 — adaptive_jbu depth_thresh sweep x4 n=100 (parallel)
DTHRESHOLDS = [0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30]
custom_all = {}
methods_list = []
for dt in DTHRESHOLDS:
    name = f"ajbu_dt{str(dt).replace('.','p')}"
    custom_all[name] = _classical_method("adaptive_jbu", depth_thresh=dt)
    methods_list.append(name)

jobs = [("expC04_ajbu_depth_thresh", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C05 — Guided filter radius sweep x4 n=100

Sweep `radius` in {1, 2, 3, 5, 7, 10, 15}.


In [ ]:
#@title EXP-C05 — guided filter radius sweep x4 n=100 (parallel)
import cv2
import numpy as np
from classical.guided_filter import guided_filter
from classical.interpolation import interpolate_depth

RADII = [1, 2, 3, 5, 7, 10, 15]
custom_all = {}
methods_list = []

for r in RADII:
    name = f"guided_r{r}"
    def _make(radius):
        def fn(lr, rgb, mask, scale=4):
            from classical.jbu import to_float_rgb as _tof
            bilinear = interpolate_depth(lr, mask, scale, "bilinear")
            gray = cv2.cvtColor(_tof(rgb), cv2.COLOR_RGB2GRAY)
            return guided_filter(bilinear, gray, radius=radius)
        return fn
    custom_all[name] = _make(r)
    methods_list.append(name)

jobs = [("expC05_guided_radius", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C06 — Guided filter eps sweep x4 n=100

Sweep `eps` in {0.0001, 0.001, 0.0025, 0.005, 0.01, 0.05, 0.1}.


In [ ]:
#@title EXP-C06 — guided filter eps sweep x4 n=100 (parallel)
import cv2
from classical.guided_filter import guided_filter
from classical.interpolation import interpolate_depth

EPS_VALS = [0.0001, 0.001, 0.0025, 0.005, 0.01, 0.05, 0.1]
custom_all = {}
methods_list = []

for eps in EPS_VALS:
    name = f"guided_eps{str(eps).replace('.','p')}"
    def _make(e):
        def fn(lr, rgb, mask, scale=4):
            from classical.jbu import to_float_rgb as _tof
            bilinear = interpolate_depth(lr, mask, scale, "bilinear")
            gray = cv2.cvtColor(_tof(rgb), cv2.COLOR_RGB2GRAY)
            return guided_filter(bilinear, gray, eps=e)
        return fn
    custom_all[name] = _make(eps)
    methods_list.append(name)

jobs = [("expC06_guided_eps", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C07 — classical_blend gain sweep x4 n=100

Sweep blend `gain` in {0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0}.
gain=0 = pure back-projection bicubic, gain=1 = pure edge-sharpened dense path.


In [ ]:
#@title EXP-C07 — classical_blend gain sweep x4 n=100 (parallel)
from classical.edge_upsampling import upsample_blend_depth
from classical.hole_filling import prepare_depth
from classical.jbu import to_float_rgb as _tof

GAINS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
custom_all = {}
methods_list = []

for g in GAINS:
    name = f"blend_g{str(g).replace('.','p')}"
    def _make(gain):
        def fn(lr, rgb, mask, scale=4):
            return upsample_blend_depth(lr, mask, _tof(rgb), scale, gain=gain)
        return fn
    custom_all[name] = _make(g)
    methods_list.append(name)

jobs = [("expC07_blend_gain", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C08 — Back-projection iterations sweep x4 n=100

Sweep `iters` in {1, 2, 4, 6, 8, 12, 16, 24}.


In [ ]:
#@title EXP-C08 — back-projection iters sweep x4 n=100 (parallel)
from classical.edge_upsampling import _backproject_bicubic
from classical.hole_filling import prepare_depth
from data.preprocessing import fill_invalid_nearest

ITERS = [1, 2, 4, 6, 8, 12, 16, 24]
custom_all = {}
methods_list = []

for it in ITERS:
    name = f"backproj_i{it}"
    def _make(iters):
        def fn(lr, rgb, mask, scale=4):
            import numpy as np
            depth, valid = prepare_depth(lr, mask)
            filled = depth if valid.all() else fill_invalid_nearest(depth, valid)
            return _backproject_bicubic(filled, scale, iters=iters)
        return fn
    custom_all[name] = _make(it)
    methods_list.append(name)

jobs = [("expC08_backproj_iters", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C09 — Back-projection gain sweep x4 n=100

Sweep correction `gain` in {0.5, 0.7, 0.9, 1.0, 1.1, 1.2, 1.5}.


In [ ]:
#@title EXP-C09 — back-projection gain sweep x4 n=100 (parallel)
from classical.edge_upsampling import _backproject_bicubic
from classical.hole_filling import prepare_depth
from data.preprocessing import fill_invalid_nearest

GAINS = [0.5, 0.7, 0.9, 1.0, 1.1, 1.2, 1.5]
custom_all = {}
methods_list = []

for g in GAINS:
    name = f"backproj_g{str(g).replace('.','p')}"
    def _make(gain):
        def fn(lr, rgb, mask, scale=4):
            depth, valid = prepare_depth(lr, mask)
            filled = depth if valid.all() else fill_invalid_nearest(depth, valid)
            return _backproject_bicubic(filled, scale, gain=gain)
        return fn
    custom_all[name] = _make(g)
    methods_list.append(name)

jobs = [("expC09_backproj_gain", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C10 — Boundary refinement relative_jump sweep x4 n=100

Sweep `relative_jump` in {0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20}.


In [ ]:
#@title EXP-C10 — boundary refinement relative_jump sweep x4 n=100 (parallel)
from classical.boundary_refinement import refine_boundaries
from classical.jbu import joint_bilateral_upsample, to_float_rgb as _tof

JUMPS = [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]
custom_all = {}
methods_list = []

for j in JUMPS:
    name = f"refined_jbu_rj{str(j).replace('.','p')}"
    def _make(rj):
        def fn(lr, rgb, mask, scale=4):
            rgb_f = _tof(rgb)
            out = joint_bilateral_upsample(lr, mask, rgb_f, radius=1, use_depth=True)
            return refine_boundaries(out, out, rgb_f, relative_jump=rj)
        return fn
    custom_all[name] = _make(j)
    methods_list.append(name)

jobs = [("expC10_boundary_rj", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C11 — Boundary refinement sigma_color sweep x4 n=100

Sweep `sigma_color` in {0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50}.


In [ ]:
#@title EXP-C11 — boundary refinement sigma_color sweep x4 n=100 (parallel)
from classical.boundary_refinement import refine_boundaries
from classical.jbu import joint_bilateral_upsample, to_float_rgb as _tof

SIGMAS = [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]
custom_all = {}
methods_list = []

for s in SIGMAS:
    name = f"refined_jbu_sc{str(s).replace('.','p')}"
    def _make(sc):
        def fn(lr, rgb, mask, scale=4):
            rgb_f = _tof(rgb)
            out = joint_bilateral_upsample(lr, mask, rgb_f, radius=1, use_depth=True)
            return refine_boundaries(out, out, rgb_f, sigma_color=sc)
        return fn
    custom_all[name] = _make(s)
    methods_list.append(name)

jobs = [("expC11_boundary_sigma", methods_list, custom_all, 4,
         DegradationConfig(), 100, None)]
res = run_classical_parallel(jobs)
for tag, md_str in res.items():
    print(f"\n--- {tag} ---"); print(md_str)


## EXP-C12/C13/C14 — All classical methods x2 / x8 / x16 n=200 (parallel)


In [ ]:
#@title EXP-C12/C13/C14 — all classical methods x2/x8/x16 n=200 (parallel)
ALL_CLASSICAL = ["nearest","bilinear","bicubic","jbu","guided",
                 "classical","classical_blend","adaptive_jbu","refined_jbu"]

jobs = [
    ("expC12_all_x2",  ALL_CLASSICAL, {}, 2, DegradationConfig(), 200, None),
    ("expC13_all_x8",  ALL_CLASSICAL, {}, 8, DegradationConfig(), 200, None),
    ("expC14_all_x16", ALL_CLASSICAL, {}, 16, DegradationConfig(), 200, None),
]
res = run_classical_parallel(jobs, max_workers=3)
for tag in ("expC12_all_x2","expC13_all_x8","expC14_all_x16"):
    print(f"\n=== {tag} ==="); print(res.get(tag,"not run"))


## EXP-C15/C16 — All classical methods x4 holes 10% and 20% n=100 (parallel)


In [ ]:
#@title EXP-C15/C16 — all classical x4 holes 10%+20% n=100 (parallel)
ALL_CLASSICAL = ["nearest","bilinear","bicubic","jbu","guided",
                 "classical","classical_blend","adaptive_jbu","refined_jbu"]

jobs = [
    ("expC15_all_holes10", ALL_CLASSICAL, {}, 4,
     DegradationConfig(hole_ratio=0.10), 100, None),
    ("expC16_all_holes20", ALL_CLASSICAL, {}, 4,
     DegradationConfig(hole_ratio=0.20), 100, None),
]
res = run_classical_parallel(jobs, max_workers=2)
for tag in ("expC15_all_holes10","expC16_all_holes20"):
    print(f"\n=== {tag} ==="); print(res.get(tag,"not run"))


## EXP-C17/C18 — All classical methods x4 noise 0.02 and 0.05 n=100 (parallel)


In [ ]:
#@title EXP-C17/C18 — all classical x4 noise 0.02+0.05 n=100 (parallel)
ALL_CLASSICAL = ["nearest","bilinear","bicubic","jbu","guided",
                 "classical","classical_blend","adaptive_jbu","refined_jbu"]

jobs = [
    ("expC17_all_noise002", ALL_CLASSICAL, {}, 4,
     DegradationConfig(noise_std=0.02), 100, None),
    ("expC18_all_noise005", ALL_CLASSICAL, {}, 4,
     DegradationConfig(noise_std=0.05), 100, None),
]
res = run_classical_parallel(jobs, max_workers=2)
for tag in ("expC17_all_noise002","expC18_all_noise005"):
    print(f"\n=== {tag} ==="); print(res.get(tag,"not run"))


## EXP-C19 — All classical x4 combined holes10%+noise0.02 n=100


In [ ]:
#@title EXP-C19 — all classical x4 combined holes+noise n=100
ALL_CLASSICAL = ["nearest","bilinear","bicubic","jbu","guided",
                 "classical","classical_blend","adaptive_jbu","refined_jbu"]
tag, md_str = run_classical(
    "expC19_combined_holes_noise",
    ALL_CLASSICAL, scale=4,
    deg=DegradationConfig(hole_ratio=0.10, noise_std=0.02),
    max_samples=100
)
print(md_str)


## EXP-C20 — JBU radius × scale interaction (r=1,2,3 × scale=2,4,8) n=50 (parallel)


In [ ]:
#@title EXP-C20 — JBU radius x scale interaction n=50 (parallel)
from classical.jbu import joint_bilateral_upsample, to_float_rgb as _tof

jobs = []
for scale in (2, 4, 8):
    custom_all = {}
    methods_list = []
    for r in (1, 2, 3):
        name = f"jbu_r{r}"
        def _make(radius):
            def fn(lr, rgb, mask, sc=4):
                return joint_bilateral_upsample(lr, mask, _tof(rgb), radius=radius)
            return fn
        custom_all[name] = _make(r)
        methods_list.append(name)
    methods_list = ["bicubic"] + methods_list
    jobs.append((f"expC20_jbu_radius_x{scale}", methods_list, custom_all,
                 scale, DegradationConfig(), 50, None))

res = run_classical_parallel(jobs, max_workers=3)
for scale in (2, 4, 8):
    tag = f"expC20_jbu_radius_x{scale}"
    print(f"\n=== {tag} ==="); print(res.get(tag,"not run"))


## EXP-C21 — guided+boundary_refinement pipeline variants x4 n=100

Tests: guided-only, guided+refine, jbu+refine, blend+refine, classical+refine.


In [ ]:
#@title EXP-C21 — guided+boundary pipeline variants x4 n=100 (parallel)
import cv2
from classical.guided_filter import guided_filter
from classical.boundary_refinement import refine_boundaries
from classical.jbu import joint_bilateral_upsample, to_float_rgb as _tof
from classical.interpolation import interpolate_depth
from classical.edge_upsampling import upsample_blend_depth

custom_all = {
    "guided_r3": lambda lr,rgb,m,s=4: guided_filter(
        interpolate_depth(lr,m,s,"bilinear"),
        cv2.cvtColor(_tof(rgb), cv2.COLOR_RGB2GRAY), radius=3),
    "guided_r3_refine": lambda lr,rgb,m,s=4: refine_boundaries(
        guided_filter(interpolate_depth(lr,m,s,"bilinear"),
                      cv2.cvtColor(_tof(rgb),cv2.COLOR_RGB2GRAY),radius=3),
        guided_filter(interpolate_depth(lr,m,s,"bilinear"),
                      cv2.cvtColor(_tof(rgb),cv2.COLOR_RGB2GRAY),radius=3),
        _tof(rgb)),
    "jbu_r2_refine": lambda lr,rgb,m,s=4: refine_boundaries(
        joint_bilateral_upsample(lr,m,_tof(rgb),radius=2,use_depth=True),
        joint_bilateral_upsample(lr,m,_tof(rgb),radius=2,use_depth=True),
        _tof(rgb)),
    "blend_g04_refine": lambda lr,rgb,m,s=4: refine_boundaries(
        upsample_blend_depth(lr,m,_tof(rgb),s,gain=0.4),
        upsample_blend_depth(lr,m,_tof(rgb),s,gain=0.4),
        _tof(rgb)),
}
methods_list = ["bicubic","classical_blend"] + list(custom_all.keys())
tag, md_str = run_classical("expC21_pipeline_variants", methods_list,
                             custom=custom_all, scale=4, max_samples=100)
print(md_str)


## EXP-C22 — classical_blend gain sweep under holes 10% x4 n=100


In [ ]:
#@title EXP-C22 — blend gain sweep holes 10% x4 n=100 (parallel)
from classical.edge_upsampling import upsample_blend_depth
from classical.jbu import to_float_rgb as _tof

GAINS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
custom_all = {}
methods_list = ["bicubic"]
for g in GAINS:
    name = f"blend_g{str(g).replace('.','p')}"
    def _make(gain):
        def fn(lr, rgb, mask, scale=4):
            return upsample_blend_depth(lr, mask, _tof(rgb), scale, gain=gain)
        return fn
    custom_all[name] = _make(g)
    methods_list.append(name)

tag, md_str = run_classical("expC22_blend_holes10", methods_list,
    custom=custom_all, scale=4,
    deg=DegradationConfig(hole_ratio=0.10), max_samples=100)
print(md_str)


## EXP-C23 — classical_blend gain sweep under noise 0.02 x4 n=100


In [ ]:
#@title EXP-C23 — blend gain sweep noise 0.02 x4 n=100 (parallel)
from classical.edge_upsampling import upsample_blend_depth
from classical.jbu import to_float_rgb as _tof

GAINS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
custom_all = {}
methods_list = ["bicubic"]
for g in GAINS:
    name = f"blend_g{str(g).replace('.','p')}"
    def _make(gain):
        def fn(lr, rgb, mask, scale=4):
            return upsample_blend_depth(lr, mask, _tof(rgb), scale, gain=gain)
        return fn
    custom_all[name] = _make(g)
    methods_list.append(name)

tag, md_str = run_classical("expC23_blend_noise002", methods_list,
    custom=custom_all, scale=4,
    deg=DegradationConfig(noise_std=0.02), max_samples=100)
print(md_str)


## EXP-C24 — JBU color_thresh × depth_thresh grid x4 n=50

3×3 grid: color_thresh in {0.04,0.08,0.15} × depth_thresh in {0.05,0.10,0.20}.


In [ ]:
#@title EXP-C24 — JBU color_thresh x depth_thresh grid x4 n=50 (parallel)
from classical.jbu import joint_bilateral_upsample, to_float_rgb as _tof

CT_VALS = [0.04, 0.08, 0.15]
DT_VALS = [0.05, 0.10, 0.20]
custom_all = {}
methods_list = ["bicubic"]
for ct in CT_VALS:
    for dt in DT_VALS:
        name = f"ajbu_ct{str(ct).replace('.','p')}_dt{str(dt).replace('.','p')}"
        def _make(c, d):
            def fn(lr, rgb, mask, scale=4):
                return joint_bilateral_upsample(lr, mask, _tof(rgb),
                    radius=1, color_thresh=c, use_depth=True, depth_thresh=d)
            return fn
        custom_all[name] = _make(ct, dt)
        methods_list.append(name)

tag, md_str = run_classical("expC24_jbu_grid", methods_list,
    custom=custom_all, scale=4, max_samples=50)
print(md_str)


## EXP-C25 — fill_small_holes relative_span sweep x4 n=100

Sweep `relative_span` in {0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20}.


In [ ]:
#@title EXP-C25 — fill_small_holes relative_span sweep x4 n=100 (parallel)
from classical.hole_filling import fill_small_holes, prepare_depth
from classical.edge_upsampling import upsample_blend_depth
from classical.jbu import to_float_rgb as _tof

SPANS = [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]
custom_all = {}
methods_list = ["bicubic", "classical_blend"]
for sp in SPANS:
    name = f"blend_fsh_sp{str(sp).replace('.','p')}"
    def _make(span):
        def fn(lr, rgb, mask, scale=4):
            depth, valid = prepare_depth(lr, mask)
            depth_f, valid_f = fill_small_holes(depth, valid, relative_span=span)
            return upsample_blend_depth(depth_f, valid_f, _tof(rgb), scale, gain=0.4)
        return fn
    custom_all[name] = _make(sp)
    methods_list.append(name)

tag, md_str = run_classical("expC25_fill_span", methods_list,
    custom=custom_all, scale=4,
    deg=DegradationConfig(hole_ratio=0.10), max_samples=100)
print(md_str)


## EXP-C26 — All classical methods full val n=654 x4 clean (definitive numbers)


In [ ]:
#@title EXP-C26 — all classical full val n=654 x4 clean
ALL_CLASSICAL = ["nearest","bilinear","bicubic","jbu","guided",
                 "classical","classical_blend","adaptive_jbu","refined_jbu"]
tag, md_str = run_classical("expC26_all_fullval_x4", ALL_CLASSICAL,
    scale=4, max_samples=654)
print(md_str)


## EXP-C27 — Per-scale best classical vs bicubic x2/x4/x8/x16 n=200 (parallel)


In [ ]:
#@title EXP-C27 — best classical vs bicubic x2/x4/x8/x16 n=200 (parallel)
BEST = ["bicubic", "classical_blend", "refined_jbu"]
jobs = [
    (f"expC27_best_x{s}", BEST, {}, s, DegradationConfig(), 200, None)
    for s in (2, 4, 8, 16)
]
res = run_classical_parallel(jobs, max_workers=4)
for s in (2, 4, 8, 16):
    tag = f"expC27_best_x{s}"
    print(f"\n=== x{s} ==="); print(res.get(tag,"not run"))


## EXP-C28 — Holes sweep 5/10/20/30/50% best classical x4 n=100 (parallel)


In [ ]:
#@title EXP-C28 — holes sweep best classical x4 n=100 (parallel)
BEST = ["bicubic", "classical_blend", "refined_jbu", "jbu"]
jobs = [
    (f"expC28_holes{int(r*100)}pct", BEST, {}, 4,
     DegradationConfig(hole_ratio=r), 100, None)
    for r in (0.05, 0.10, 0.20, 0.30, 0.50)
]
res = run_classical_parallel(jobs, max_workers=4)
for r in (0.05, 0.10, 0.20, 0.30, 0.50):
    tag = f"expC28_holes{int(r*100)}pct"
    print(f"\n=== holes {int(r*100)}% ==="); print(res.get(tag,"not run"))


## EXP-C29 — Noise sweep 0.01/0.02/0.05/0.10/0.20 best classical x4 n=100 (parallel)


In [ ]:
#@title EXP-C29 — noise sweep best classical x4 n=100 (parallel)
BEST = ["bicubic", "classical_blend", "refined_jbu", "jbu"]
jobs = [
    (f"expC29_noise{str(s).replace('.','p')}", BEST, {}, 4,
     DegradationConfig(noise_std=s), 100, None)
    for s in (0.01, 0.02, 0.05, 0.10, 0.20)
]
res = run_classical_parallel(jobs, max_workers=4)
for s in (0.01, 0.02, 0.05, 0.10, 0.20):
    tag = f"expC29_noise{str(s).replace('.','p')}"
    print(f"\n=== noise {s} ==="); print(res.get(tag,"not run"))


## EXP-C30 — Combined degradation grid (holes × noise) best classical x4 n=50 (parallel)

3×3 grid: holes in {0.05,0.10,0.20} × noise in {0.01,0.02,0.05}.


In [ ]:
#@title EXP-C30 — combined degradation grid x4 n=50 (parallel)
BEST = ["bicubic", "classical_blend", "refined_jbu"]
jobs = []
for hr in (0.05, 0.10, 0.20):
    for ns in (0.01, 0.02, 0.05):
        tag = f"expC30_h{int(hr*100)}_n{str(ns).replace('.','p')}"
        jobs.append((tag, BEST, {}, 4,
                     DegradationConfig(hole_ratio=hr, noise_std=ns), 50, None))

res = run_classical_parallel(jobs, max_workers=4)
for tag in sorted(res):
    print(f"\n=== {tag} ==="); print(res[tag])


## FINAL DUMP — All results in one self-contained JSON block

Run this cell last. Prints a single fenced JSON block with every experiment's
summary + per-image rows + run environment. Readable directly from the `.ipynb`.


In [ ]:
#@title FINAL DUMP — all results as one JSON block
import glob, json, csv, os, sys, platform, subprocess, torch
from pathlib import Path

all_results = {}
for json_path in sorted(glob.glob("results/**/*summary.json", recursive=True)):
    tag  = Path(json_path).parent.name
    data = json.loads(open(json_path).read())
    all_results[tag] = data

per_image = {}
for csv_path in sorted(glob.glob("results/**/*per_image.csv", recursive=True)):
    tag = Path(csv_path).parent.name
    with open(csv_path, newline="") as f:
        per_image[tag] = list(csv.DictReader(f))

env = {
    "python":   sys.version.split()[0],
    "torch":    torch.__version__,
    "cuda":     torch.cuda.is_available(),
    "n_gpu":    torch.cuda.device_count(),
    "gpus":     [torch.cuda.get_device_name(i)
                 for i in range(torch.cuda.device_count())],
    "platform": platform.platform(),
}
try:
    env["branch"] = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
    env["commit"] = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"]).decode().strip()
except Exception:
    pass

dump = {"run_env": env, "experiments": all_results, "per_image": per_image}

print("```json")
print(json.dumps(dump, indent=2))
print("```")

out_path = "results/kaggle_ALL_RESULTS_DUMP.json"
Path(out_path).write_text(json.dumps(dump, indent=2))
print(f"\nSaved: {out_path}")
print(f"Experiments: {len(all_results)}  |  per-image rows: {sum(len(v) for v in per_image.values())}")
